In [7]:
import sys, platform, subprocess, datetime
import torch

print("===== Environment Info =====")
print("OS:", platform.platform())
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA (torch built with):", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU compute capability:", torch.cuda.get_device_capability(0))
    print("GPU total memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

try:
    import transformers
    print("transformers:", transformers.__version__)
except ImportError:
    print("transformers: not installed")

print("\n--- nvidia-smi (driver version included")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

===== Environment Info =====
OS: Linux-6.6.122+-x86_64-with-glibc2.35
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA (torch built with): 12.8
cuDNN version: 91900
GPU available: True
GPU name: Tesla T4
GPU compute capability: (7, 5)
GPU total memory (GB): 15.637086208
transformers: 5.13.1

--- nvidia-smi (driver version included
Tue Aug 18 02:37:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+===

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/KV Cache/0_KV_Cache_verification/"

In [9]:
!python fp32_test_file.py

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
config.json: 100% 665/665 [00:00<00:00, 3.70MB/s]

model.safetensors: downloading bytes:  31% 169M/548M [00:01<00:01, 239MB/s, 11.9MB/s  ]
model.safetensors: downloading bytes:  43% 233M/548M [00:01<00:01, 271MB/s, 19.7MB/s  ]
model.safetensors: downloading bytes:  58% 319M/548M [00:01<00:00, 237MB/s, 27.9MB/s  ]
model.safetensors: downloading bytes:  87% 474M/548M [00:02<00:00, 261MB/s, 41.8MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:02<00:00, 174MB/s, 42.5MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:02<00:00, 201MB/s, 49.7MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 7753.55it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 552kB/s]
===== running_mode = inf_no_cache =====
[inf_no_cache] step 3: next tokens = [' The', ' get', ' action']
[inf_no_cache] step 4: next tokens = [' F

In [10]:
!python fp64_test_file.py

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
Loading weights: 100% 148/148 [00:00<00:00, 771.69it/s]
===== running_mode = inf_no_cache =====
[inf_no_cache] step 3: next tokens = [' The', ' get', ' action']
[inf_no_cache] step 4: next tokens = [' First', ' to', ' against']
[inf_no_cache] step 5: next tokens = [' Time', ' the', ' the']
[inf_no_cache] step 6: next tokens = [' You', ' actual', ' company']
[inf_no_cache] step 7: next tokens = [' Can', ' game', '.']
[inf_no_cache] step 8: next tokens = [' Play', ',', '\n']
[inf_no_cache] step 9: next tokens = [' The', ' let', '\n']
[inf_no_cache] step 10: next tokens = [' First', "'s", 'The']
[inf_no_cache] step 11: next tokens = [' Time', ' talk', ' company']
[inf_no_cache] step 12: next tokens = [' You', ' about', ' has']
===== running_mode = inf_cache =====
[inf_cache] step 3: next tokens = [' The', ' get', ' action']
[inf_cache]

# KV value error check
- Activation
- KV Cache

In [11]:
import torch

def KV_values(dtype:str, first_token:int, last_token:int):
    max_K_diff_val = torch.tensor(0.0, dtype=torch.float64)
    max_K_diff_loc = {'layer': -1, 'batch': -1, 'token': -1}

    max_V_diff_val = torch.tensor(0.0, dtype=torch.float64)
    max_V_diff_loc = {'layer': -1, 'batch': -1, 'token': -1}

    for layer in range(12):
      print("\n\n================================================= Layer:",layer+1)
      data_cache_used = torch.load(f"./KV_cache_result/{dtype}_{layer}layer__{last_token}token__inf_cache.pt", map_location=torch.device('cpu'))
      data_cache_not_used = torch.load(f"./KV_cache_result/{dtype}_{layer}layer__{last_token}token__inf_no_cache.pt", map_location=torch.device('cpu'))

      for _batch in range(1,4):
        batch = _batch-1
        for _token in range(first_token, last_token):
          token = _token-1
          print(f"K cache - Batch: {_batch} - Token: {_token}",torch.allclose(data_cache_used['K_cache'][batch][token], data_cache_not_used['K_cache'][batch][token], rtol=1e-05, atol=1e-05))
          print(f"V cache - Batch: {_batch} - Token: {_token}",torch.allclose(data_cache_used['V_cache'][batch][token], data_cache_not_used['V_cache'][batch][token], rtol=1e-05, atol=1e-05))

          K_diff = (data_cache_used['K_cache'][batch][token] - data_cache_not_used['K_cache'][batch][token]).abs()
          K_diff_top3_values, K_top3_indices = torch.topk(K_diff.flatten(), 3)
          print(f"Samples of K cache:", K_diff_top3_values)
          if K_diff_top3_values[0] > max_K_diff_val:
            max_K_diff_val = K_diff_top3_values[0]
            max_K_diff_loc = {'layer': layer + 1, 'batch': _batch, 'token': _token}

          V_diff = (data_cache_used['V_cache'][batch][token] - data_cache_not_used['V_cache'][batch][token]).abs()
          V_diff_top3_values, V_top3_indices = torch.topk(V_diff.flatten(), 3)
          print(f"Samples of V cache:", V_diff_top3_values)
          if V_diff_top3_values[0] > max_V_diff_val:
            max_V_diff_val = V_diff_top3_values[0]
            max_V_diff_loc = {'layer': layer + 1, 'batch': _batch, 'token': _token}
          print("\n\n")

    print("\n\n=================================================")
    print(f"Overall Max K_diff Value: {max_K_diff_val} at Layer: {max_K_diff_loc['layer']}, Batch: {max_K_diff_loc['batch']}, Token: {max_K_diff_loc['token']}")
    print(f"Overall Max V_diff Value: {max_V_diff_val} at Layer: {max_V_diff_loc['layer']}, Batch: {max_V_diff_loc['batch']}, Token: {max_V_diff_loc['token']}")
    return max_K_diff_val, max_V_diff_val

In [12]:
KV_values(dtype = "float32", first_token=3, last_token=12)



================================================= Layer: 1
K cache - Batch: 1 - Token: 3 True
V cache - Batch: 1 - Token: 3 True
Samples of K cache: tensor([0., 0., 0.])
Samples of V cache: tensor([0., 0., 0.])



K cache - Batch: 1 - Token: 4 True
V cache - Batch: 1 - Token: 4 True
Samples of K cache: tensor([9.5367e-07, 9.5367e-07, 9.5367e-07])
Samples of V cache: tensor([3.5763e-07, 2.3842e-07, 2.3842e-07])



K cache - Batch: 1 - Token: 5 True
V cache - Batch: 1 - Token: 5 True
Samples of K cache: tensor([1.4305e-06, 1.0729e-06, 9.5367e-07])
Samples of V cache: tensor([2.3842e-07, 2.2352e-07, 1.7881e-07])



K cache - Batch: 1 - Token: 6 True
V cache - Batch: 1 - Token: 6 True
Samples of K cache: tensor([1.4305e-06, 9.5367e-07, 9.5367e-07])
Samples of V cache: tensor([2.0862e-07, 2.0862e-07, 1.7881e-07])



K cache - Batch: 1 - Token: 7 True
V cache - Batch: 1 - Token: 7 True
Samples of K cache: tensor([9.5367e-07, 9.5367e-07, 9.5367e-07])
Samples of V cache: tensor([2.3842e-07, 

(tensor(1.4305e-05), tensor(7.0333e-06))

In [13]:
KV_values(dtype = "float64", first_token=3, last_token=12)



================================================= Layer: 1
K cache - Batch: 1 - Token: 3 True
V cache - Batch: 1 - Token: 3 True
Samples of K cache: tensor([8.8818e-15, 8.8818e-15, 5.7732e-15], dtype=torch.float64)
Samples of V cache: tensor([1.5543e-15, 1.1102e-15, 1.1102e-15], dtype=torch.float64)



K cache - Batch: 1 - Token: 4 True
V cache - Batch: 1 - Token: 4 True
Samples of K cache: tensor([7.1054e-15, 6.2172e-15, 5.3291e-15], dtype=torch.float64)
Samples of V cache: tensor([2.6645e-15, 2.2204e-15, 1.2212e-15], dtype=torch.float64)



K cache - Batch: 1 - Token: 5 True
V cache - Batch: 1 - Token: 5 True
Samples of K cache: tensor([4.4409e-15, 4.4409e-15, 4.4409e-15], dtype=torch.float64)
Samples of V cache: tensor([1.1102e-15, 9.9920e-16, 9.9920e-16], dtype=torch.float64)



K cache - Batch: 1 - Token: 6 True
V cache - Batch: 1 - Token: 6 True
Samples of K cache: tensor([1.0658e-14, 6.2172e-15, 6.2172e-15], dtype=torch.float64)
Samples of V cache: tensor([1.3323e-15, 1.1102e-

(tensor(4.9738e-14, dtype=torch.float64),
 tensor(2.4869e-14, dtype=torch.float64))

## Difference between fp64 and fp32

In [14]:
import torch

def analyze_error_differences(first_token: int, last_token: int):
    # Initialize overall max/min ratios with extreme values
    overall_max_K_ratio_of_errors = torch.tensor(float('-inf'), dtype=torch.float64)
    overall_min_K_ratio_of_errors = torch.tensor(float('inf'), dtype=torch.float64)
    overall_max_V_ratio_of_errors = torch.tensor(float('-inf'), dtype=torch.float64)
    overall_min_V_ratio_of_errors = torch.tensor(float('inf'), dtype=torch.float64)

    # Initialize location trackers for max/min ratios
    max_K_loc = {'max_val': float('-inf'), 'layer': -1, 'batch': -1, 'token': -1}
    min_K_loc = {'min_val': float('inf'), 'layer': -1, 'batch': -1, 'token': -1}
    max_V_loc = {'max_val': float('-inf'), 'layer': -1, 'batch': -1, 'token': -1}
    min_V_loc = {'min_val': float('inf'), 'layer': -1, 'batch': -1, 'token': -1}

    # Counters for skipped items
    skipped_K_count = 0
    skipped_V_count = 0
    total_K_entries = 0
    total_V_entries = 0

    print("Starting error ratio analysis (skipping zero errors)...")

    for layer in range(12):
        print(f"Processing Layer: {layer + 1}")
        # Load float32 caches for the current layer
        data_cache_used_fp32 = torch.load(f"./KV_cache_result/float32_{layer}layer__{last_token}token__inf_cache.pt", map_location=torch.device('cpu'))
        data_cache_not_used_fp32 = torch.load(f"./KV_cache_result/float32_{layer}layer__{last_token}token__inf_no_cache.pt", map_location=torch.device('cpu'))

        # Load float64 caches for the current layer
        data_cache_used_fp64 = torch.load(f"./KV_cache_result/float64_{layer}layer__{last_token}token__inf_cache.pt", map_location=torch.device('cpu'))
        data_cache_not_used_fp64 = torch.load(f"./KV_cache_result/float64_{layer}layer__{last_token}token__inf_no_cache.pt", map_location=torch.device('cpu'))

        for _batch in range(1, 4): # Batch indices 0, 1, 2
            batch = _batch - 1
            for _token in range(first_token, last_token): # Token indices from first_token-1 to last_token-1
                token = _token - 1

                # Calculate max K_diff for float32 for current (layer, batch, token)
                K_diff_fp32_current = (data_cache_used_fp32['K_cache'][batch][token] - data_cache_not_used_fp32['K_cache'][batch][token]).abs().max()
                # Calculate max V_diff for float32 for current (layer, batch, token)
                V_diff_fp32_current = (data_cache_used_fp32['V_cache'][batch][token] - data_cache_not_used_fp32['V_cache'][batch][token]).abs().max()

                # Calculate max K_diff for float64 for current (layer, batch, token)
                K_diff_fp64_current = (data_cache_used_fp64['K_cache'][batch][token] - data_cache_not_used_fp64['K_cache'][batch][token]).abs().max()
                # Calculate max V_diff for float64 for current (layer, batch, token)
                V_diff_fp64_current = (data_cache_used_fp64['V_cache'][batch][token] - data_cache_not_used_fp64['V_cache'][batch][token]).abs().max()

                # Process K_cache ratio
                total_K_entries += 1
                if K_diff_fp32_current == 0 or K_diff_fp64_current == 0:
                    skipped_K_count += 1
                else:
                    ratio_K_current = K_diff_fp64_current / K_diff_fp32_current
                    # Update overall max/min for K ratios
                    if ratio_K_current > overall_max_K_ratio_of_errors:
                        overall_max_K_ratio_of_errors = ratio_K_current
                        max_K_loc = {'max_val': ratio_K_current.item(), 'layer': layer + 1, 'batch': _batch, 'token': _token}
                    if ratio_K_current < overall_min_K_ratio_of_errors:
                        overall_min_K_ratio_of_errors = ratio_K_current
                        min_K_loc = {'min_val': ratio_K_current.item(), 'layer': layer + 1, 'batch': _batch, 'token': _token}

                # Process V_cache ratio
                total_V_entries += 1
                if V_diff_fp32_current == 0 or V_diff_fp64_current == 0:
                    skipped_V_count += 1
                else:
                    ratio_V_current = V_diff_fp64_current / V_diff_fp32_current
                    # Update overall max/min for V ratios
                    if ratio_V_current > overall_max_V_ratio_of_errors:
                        overall_max_V_ratio_of_errors = ratio_V_current
                        max_V_loc = {'max_val': ratio_V_current.item(), 'layer': layer + 1, 'batch': _batch, 'token': _token}
                    if ratio_V_current < overall_min_V_ratio_of_errors:
                        overall_min_V_ratio_of_errors = ratio_V_current
                        min_V_loc = {'min_val': ratio_V_current.item(), 'layer': layer + 1, 'batch': _batch, 'token': _token}

    print("\n=================================================")
    print("Overall Max/Min of (fp64_MaxError / fp32_MaxError) per (Layer, Batch, Token) [Skipping zero errors]:")
    print(f"  K_cache: Max Ratio: {overall_max_K_ratio_of_errors:.4e} at Layer: {max_K_loc['layer']}, Batch: {max_K_loc['batch']}, Token: {max_K_loc['token']}")
    print(f"           Min Ratio: {overall_min_K_ratio_of_errors:.4e} at Layer: {min_K_loc['layer']}, Batch: {min_K_loc['batch']}, Token: {min_K_loc['token']}")
    print(f"           Skipped {skipped_K_count} out of {total_K_entries} K-cache entries ({skipped_K_count / total_K_entries * 100:.2f}%)")
    print(f"  V_cache: Max Ratio: {overall_max_V_ratio_of_errors:.4e} at Layer: {max_V_loc['layer']}, Batch: {max_V_loc['batch']}, Token: {max_V_loc['token']}")
    print(f"           Min Ratio: {overall_min_V_ratio_of_errors:.4e} at Layer: {min_V_loc['layer']}, Batch: {min_V_loc['batch']}, Token: {min_V_loc['token']}")
    print(f"           Skipped {skipped_V_count} out of {total_V_entries} V-cache entries ({skipped_V_count / total_V_entries * 100:.2f}%)")

    return overall_max_K_ratio_of_errors, overall_min_K_ratio_of_errors, \
           overall_max_V_ratio_of_errors, overall_min_V_ratio_of_errors

In [15]:
print("\n\n=================================================")
print("Executing granular error difference analysis...")
max_K_diff_of_diffs, min_K_diff_of_diffs, max_V_diff_of_diffs, min_V_diff_of_diffs = analyze_error_differences(first_token=3, last_token=12)



Executing granular error difference analysis...
Starting error ratio analysis (skipping zero errors)...
Processing Layer: 1
Processing Layer: 2
Processing Layer: 3
Processing Layer: 4
Processing Layer: 5
Processing Layer: 6
Processing Layer: 7
Processing Layer: 8
Processing Layer: 9
Processing Layer: 10
Processing Layer: 11
Processing Layer: 12

Overall Max/Min of (fp64_MaxError / fp32_MaxError) per (Layer, Batch, Token) [Skipping zero errors]:
  K_cache: Max Ratio: 1.4901e-08 at Layer: 1, Batch: 2, Token: 5
           Min Ratio: 1.0865e-09 at Layer: 8, Batch: 2, Token: 5
           Skipped 3 out of 324 K-cache entries (0.93%)
  V_cache: Max Ratio: 1.4901e-08 at Layer: 1, Batch: 1, Token: 10
           Min Ratio: 1.3767e-09 at Layer: 5, Batch: 3, Token: 6
           Skipped 3 out of 324 V-cache entries (0.93%)
